<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Código suplementar do livro <a href="https://mng.bz/lZ5B">Build a Reasoning Model (From Scratch)</a>, de <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Repositório de código: <a href="https://github.com/rasbt/reasoning-from-scratch">https://github.com/rasbt/reasoning-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="https://mng.bz/lZ5B"><img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

<!-- aviso-traducao-ptbr -->
<sub>
<b>Tradução não oficial para português do Brasil.</b> Este arquivo é uma obra
derivada do repositório original de Sebastian Raschka
(<a href="https://github.com/rasbt/reasoning-from-scratch">rasbt/reasoning-from-scratch</a>),
licenciado sob Apache License 2.0. Apenas o texto foi traduzido; o código
permanece inalterado. Não é uma publicação oficial da Manning e não substitui o
livro. Detalhes das convenções em <code>GLOSSARIO-TRADUCAO.md</code>.
</sub>

# Capítulo 3: Soluções dos exercícios

Pacotes usados neste notebook:

In [1]:
from importlib.metadata import version

used_libraries = [
    "reasoning_from_scratch",
    "torch",
    "tokenizers"  # Used by reasoning_from_scratch
]

for lib in used_libraries:
    print(f"{lib} version: {version(lib)}")

reasoning_from_scratch version: 0.1.4
torch version: 2.7.1
tokenizers version: 0.21.4


&nbsp;
## Exercício 3.1: Adicionando mais casos de teste

- Há um número infinito de casos de teste diferentes que podemos adicionar
- Abaixo está uma seleção de alguns interessantes

In [ ]:
from reasoning_from_scratch.ch03 import (
    run_demos_table
)

more_tests = [
    # Different bracket types
    ("check_17", "[1, 2]", "(1, 2)", True),

    # Scientific notation
    ("check_18", "1e-3", "0.001", True),

    # Algebraic simplification with caret exponent
    ("check_19", "(-3)^2", "9", True),

    # Unicode minus (U+2212) vs ASCII hyphen-minus
    ("check_20", "−1", "-1", True),

]

run_demos_table(more_tests)

Test     | Expect | Got   | Status
check_17 | True   | True  | PASS  
check_18 | True   | True  | PASS  
check_19 | True   | True  | PASS  
check_20 | True   | False | FAIL  

Passed 3/4


- Como podemos ver, os testes passam em todos os casos, exceto no `check_20`, que troca o sinal comum por uma versão Unicode do sinal de menos, indistinguível a olho nu
- Poderíamos corrigir esse caso de teste adicionando uma das linhas a seguir em qualquer ponto da função `normalize_text`

```python
text = text.replace("−", "-")
# or
text = text.replace("\u2212", "-")
```

- À primeira vista, outro teste interessante é o seguinte:

In [2]:
extra_tests_1 = [
    ("check_21", "Text around answer 3.", "3", True)
]

run_demos_table(extra_tests_1)

Test     | Expect | Got   | Status
check_21 | True   | False | FAIL  

Passed 0/1


- Embora possa parecer que nosso código não consegue lidar com esses casos que contêm texto, na verdade esse é um teste mal projetado
- Na prática, a função `run_demos_table` serve especificamente para testar a função `grade_answer`; nada mais, nada menos
- A função `grade_answer` nunca receberia a resposta inteira nesse formato, já que a resposta teria sido extraída do texto antes de ser passada para ela

Ou seja, se quisermos testar respostas em texto, precisamos chamar o teste da seguinte forma:

In [3]:
from reasoning_from_scratch.ch03 import (
    extract_final_candidate
)


extra_tests_2 = [
    ("check_21",
     extract_final_candidate("Text around answer 3."),
     "3", True)
]
run_demos_table(extra_tests_2)

Test     | Expect | Got  | Status
check_21 | True   | True | PASS  

Passed 1/1


&nbsp;
## Exercício 3.2: Calculando o comprimento médio das respostas

- Opção A: poderíamos modificar a função `evaluate_math500_stream` adicionando as seguintes linhas:

```python
# ...
# below `num_correct = 0`
total_len = 0

# ...
# inside for i, row in enumerate(math_data, start=1):
# anywhere below `gen_text = ...`
total_len += len(tokenizer.encode(gen_text))

# ...
# anywhere at the bottom before the return statement
avg_len = total_len / num_examples
print(f"Average length: {avg_len:.2f} tokens")
```

- Como alternativa, também podemos calcular os comprimentos das respostas a partir dos arquivos `.jsonl` que foram criados quando rodamos a função `evaluate_math500_stream` no capítulo principal
- Primeiro, carregamos o arquivo `.jsonl` da seguinte forma:

In [5]:
import json
from pathlib import Path

WHICH_MODEL = "base"

dev_name = "mps"  # e.g., "cuda", "cpu"

# You may need to adjust this path:
local_path = Path(f"math500-{dev_name}.jsonl")
if not local_path.exists():
    raise FileNotFoundError(
        f"{local_path} not found. Run ch03_main.ipynb to create it."
    )

results = []
with open(local_path, "r") as f:
    for line in f:
        if line.strip():
            results.append(json.loads(line))

print("Number of entries:", len(results))


Number of entries: 10


- Note que cada entrada tem várias chaves; no entanto, só nos interessa a chave `"generated_text"`, que contém a resposta completa do modelo:

In [6]:
print(results[0].keys())

dict_keys(['index', 'problem', 'gtruth_answer', 'generated_text', 'extracted', 'correct'])


- Note que cada entrada tem várias chaves; no entanto, só nos interessa a chave `"generated_text"`, que contém a resposta completa do modelo:

In [7]:
from reasoning_from_scratch.qwen3 import (
    download_qwen3_small,
    Qwen3Tokenizer
)

if WHICH_MODEL == "base":

    download_qwen3_small(
        kind="base", tokenizer_only=True, out_dir="qwen3"
    )
    tokenizer_path = Path("qwen3") / "tokenizer-base.json"
    tokenizer = Qwen3Tokenizer(tokenizer_file_path=tokenizer_path)

elif WHICH_MODEL == "reasoning":

    download_qwen3_small(
        kind="reasoning", tokenizer_only=True, out_dir="qwen3"
    )
    tokenizer_path = Path("qwen3") / "tokenizer-reasoning.json"
    tokenizer = Qwen3Tokenizer(
        tokenizer_file_path=tokenizer_path,
        apply_chat_template=True,
        add_generation_prompt=True,
        add_thinking=True,
    )

✓ qwen3/tokenizer-base.json already up-to-date


- Em seguida, podemos calcular o comprimento médio da forma a seguir, que é parecida com o modo como poderíamos ter modificado a função `evaluate_math500_stream`:

In [8]:
total_len = 0

for item in results:
    num_tokens = len(tokenizer.encode(item["generated_text"]))
    total_len += num_tokens

avg_len = total_len / len(results)
print(f"Average length: {avg_len:.2f} tokens")

Average length: 98.00 tokens


| Modo      | Dispositivo | Comprimento médio | Tamanho do MATH-500 |
|-----------|---------|----------------|----------------|
| Base      | CPU     | 97,3           | 10             |
| Base      | MPS     | 98,0           | 10             |
| Reasoning | CPU     | 891,80         | 10             |
| Reasoning | MPS     | 1159,30        | 10             |
|           |         |                |                |
| Base      | CUDA    | 96,74          | 500            |
| Reasoning | CUDA    | 1361,21        | 500            |


- Como podemos ver, e como esperado, o modelo de raciocínio escreve respostas muito mais longas

&nbsp;
## Exercício 3.3: Estendendo ou trocando o dataset de avaliação

- Para avaliar o modelo em um dataset maior, basta trocar o `math_data[:10]` por outra fatia ou por um número maior (até 500)

```python
num_correct, num_examples, acc = evaluate_math500_stream(
    model, tokenizer, device,
    math_data=math_data[:10],
    max_new_tokens=2048,
    verbose=False
)
```

- A tabela abaixo mostra os valores de acurácia para diferentes tamanhos de dataset (como o conjunto de teste MATH-500 já vem embaralhado, nenhum embaralhamento adicional foi aplicado)

| Modo      | Dispositivo | Acurácia | Tamanho do MATH-500 |
|-----------|---------|----------|----------------|
| Base      | CUDA    | 30,0%    | 10             |
| Base      | CUDA    | 34,0%    | 50             |
| Base      | CUDA    | 27,0%    | 100            |
| Base      | CUDA    | 31,0%    | 200            |
| Base      | CUDA    | 15,3%    | 500            |
|           |         |          |                |
| Reasoning | CUDA    | 90,0%    | 10             |
| Reasoning | CUDA    | 58,0%    | 50             |
| Reasoning | CUDA    | 58,0%    | 100            |
| Reasoning | CUDA    | 56,0%    | 200            |
| Reasoning | CUDA    | 48,2%    | 500            |

- Como podemos ver pelos resultados acima, os 10 primeiros exemplos não são muito representativos do desempenho no MATH-500 avaliado sobre os 500 exemplos completos

- Além disso, podemos criar um dataset totalmente novo em estilo parecido com o do MATH-500
- Por exemplo, um dataset em estilo MATH-500 está incluído neste repositório; podemos usá-lo no capítulo principal trocando o nome do arquivo de `math500_test.json` para `math_new50_exercise.json` (este dataset está incluído no repositório GitHub deste livro, em https://github.com/rasbt/reasoning-from-scratch/tree/main/ch03/01_main-chapter-code)
- O desempenho dos modelos base e de raciocínio é o seguinte:
    - base: 36,0% (18/50)
    - reasoning: 80,0% (40/50)
- A partir disso, podemos concluir que, embora o dataset de teste MATH-500 original possa ter sido incluído no dataset de treinamento do Qwen3, o modelo mostra desempenho parecido em questões de matemática novas, o que indica que ele não está sofrendo de overfitting extenso aos dados originais do MATH-500

&nbsp;
## Exercício 3.4: Experimentando com diferentes templates de prompt

- Poderíamos usar o prompt alternativo, parecido com o sugerido no capítulo, que modifica o prompt para usar "Problem" em vez de "Question":

```python
def render_prompt(prompt):
    template = (
        "You are a helpful math assistant.\n"
        "Solve the problem and write the final result on a new line as:\n"
        "\\boxed{ANSWER}\n\n"
        f"Problem:\n{prompt}\n\nAnswer:"
    )
    return template
```

- Usar esse prompt melhora o desempenho do modelo base, nos 500 exemplos, de 15,3% para 31,2%
- E, inversamente, reduz o desempenho do modelo de raciocínio de 50,8% para 50,0%
- A partir dessas observações, podemos concluir que o modelo base é muito mais sensível ao formato do prompt (provavelmente por ter memorizado alguns exemplos do MATH-500 formatados como prompt no conjunto de treinamento) do que o modelo de raciocínio; este último parece amplamente não afetado